## Use as dataset the one given by step_0

In [2]:
import pandas as pd

candidates = pd.read_csv("candidates.csv")

sbert_results = pd.read_csv("sbert_results.csv.gz", compression="gzip")

# 2. Merge SBERT similarity scores back into candidates
df_ranked = candidates.merge(sbert_results, on="appln_id", how="left")
df_ranked = df_ranked.sort_values("sbert_similarity", ascending=False).reset_index(drop=True)

SIMILARITY_THRESHOLD = 0.60
final_dataset = df_ranked[df_ranked["sbert_similarity"] >= SIMILARITY_THRESHOLD].copy()

# 4. Print summary metrics
print(f"✅ Total baseline candidates: {len(candidates):,} patents")
print(f"🎯 Target dataset (Similarity >= {SIMILARITY_THRESHOLD*100:.0f}%): {len(final_dataset):,} patents")
if len(final_dataset) > 0:
    print(f"📊 Similarity range in target: {final_enel_dataset['sbert_similarity'].min():.4f} to {final_dataset['sbert_similarity'].max():.4f}")


✅ Total baseline candidates: 196,524 patents
🎯 Target dataset (Similarity >= 60%): 48,930 patents
📊 Similarity range in target: 0.6000 to 0.9840


In [6]:
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import PercentFormatter
from sqlalchemy import and_, case, distinct, func, literal, or_

from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import (
    TLS201_APPLN,               # import the needed tables for the extractions
    TLS202_APPLN_TITLE,
    TLS203_APPLN_ABSTR,
    TLS206_PERSON,
    TLS207_PERS_APPLN,
    TLS224_APPLN_CPC,
    TLS230_APPLN_TECHN_FIELD,
    TLS901_TECHN_FIELD_IPC,
)

In [7]:
ENV = "TEST"                 # develop on TEST, switch to "PROD" for final numbers if requested
OUTPUT_DIR = Path("outputs")

# ---------------------------------------------------------------- sector definition (SBERT Approach)
SECTOR_NAME = "Utility-Scale Energy Storage"
SIMILARITY_THRESHOLD = 0.60  # SBERT score cutoff (replaces raw CPC prefix filters)

# ---------------------------------------------------------------- time window
YEAR_MIN, YEAR_MAX = 2000, 2025

# Dating choice (PW Rules)
FAMILY_TIME_COLUMN = "earliest_filing_year"   # family level: earliest priority year
APPLN_TIME_COLUMN = "appln_filing_year"        # application level: application's filing year

# Last years incomplete in PATSTAT due to publication lag
TRUNCATED_YEARS = {"appln_filing_year": 3, "earliest_filing_year": 3, "earliest_publn_year": 1}

# ---------------------------------------------------------------- counting rules
IPR_TYPE = "PI"                                # Patents of Invention only
IP5_OFFICES = ("EP", "US", "CN", "JP", "KR")     # Major 5 patent offices
TOP_N = 8                                       # Categories shown in plots/tables

# ---------------------------------------------------------------- derived parameters
LAST_COMPLETE_YEAR = YEAR_MAX - max(TRUNCATED_YEARS[FAMILY_TIME_COLUMN], TRUNCATED_YEARS[APPLN_TIME_COLUMN])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Initialize PATSTAT client & ORM session
patstat = PatstatClient(env=ENV)
db = patstat.orm()

print(f"Sector: {SECTOR_NAME} | env: {ENV} | years {YEAR_MIN}-{YEAR_MAX} "
      f"| family year: {FAMILY_TIME_COLUMN} | application year: {APPLN_TIME_COLUMN} "
      f"| complete years up to {LAST_COMPLETE_YEAR}")

Sector: Utility-Scale Energy Storage | env: TEST | years 2000-2025 | family year: earliest_filing_year | application year: appln_filing_year | complete years up to 2022


## Selecting the sector

In [10]:
# 1. Extract application IDs to map
appln_ids = df_ranked["appln_id"].dropna().astype(int).tolist()
CHUNK_SIZE = 25000
fam_dfs = []

print(f" Fetching docdb_family_id mapping in chunks ({len(appln_ids):,} patents)...")

# 2. Query BigQuery in chunks to stay well under the 1 MB text limit
for i in range(0, len(appln_ids), CHUNK_SIZE):
    chunk_ids = appln_ids[i : i + CHUNK_SIZE]
    q_fam = db.query(
        TLS201_APPLN.appln_id,
        TLS201_APPLN.docdb_family_id
    ).filter(TLS201_APPLN.appln_id.in_(chunk_ids))
    
    fam_dfs.append(patstat.df(q_fam))

# 3. Combine chunk results into a single mapping DataFrame
fam_mapping = pd.concat(fam_dfs, ignore_index=True)

# 4. Merge mapping into df_ranked and update target set
df_ranked = df_ranked.merge(fam_mapping, on="appln_id", how="left")
final_enel_dataset = df_ranked[df_ranked["sbert_similarity"] >= SIMILARITY_THRESHOLD].copy()

print("docdb_family_id attached successfully!")
print(f"Unique target families (SBERT >= {SIMILARITY_THRESHOLD*100:.0f}%): {final_enel_dataset['docdb_family_id'].nunique():,}")

🔄 Fetching docdb_family_id mapping in chunks (196,524 patents)...
docdb_family_id attached successfully!
Unique target families (SBERT >= 60%): 40,826


## Application

In [12]:
# 1. Identify the largest patent family in our SBERT target set using pandas
family_app_counts = final_enel_dataset["docdb_family_id"].value_counts()
example_family_id = int(family_app_counts.index[0])
apps_in_target_count = family_app_counts.iloc[0]

print(f"Selected Example Family ID: {example_family_id}")
print(f"Target filings in this dataset: {apps_in_target_count} applications\n")

# 2. Query all application records for this family directly from PATSTAT
q_example_family = (
    db.query(
        TLS201_APPLN.docdb_family_id,
        TLS201_APPLN.appln_id,
        TLS201_APPLN.appln_auth,
        TLS201_APPLN.appln_kind,
        TLS201_APPLN.appln_filing_date,
        TLS201_APPLN.appln_filing_year,
        TLS201_APPLN.earliest_publn_year,
        TLS201_APPLN.earliest_filing_year,
        TLS201_APPLN.granted,
        TLS201_APPLN.docdb_family_size,
    )
    .filter(TLS201_APPLN.docdb_family_id == example_family_id)
    .order_by(TLS201_APPLN.appln_filing_date)
)

example_family = patstat.df(q_example_family)

example_family

Selected Example Family ID: 63669288
Target filings in this dataset: 33 applications



,docdb_family_id,appln_id,appln_auth,appln_kind,appln_filing_date,appln_filing_year,earliest_publn_year,earliest_filing_year,granted,docdb_family_size
0,63669288,500778272,US,A,2018-05-07,2018,2018,2016,N,107
1,63669288,500783187,US,A,2018-05-07,2018,2018,2016,Y,107
2,63669288,500782752,US,A,2018-05-07,2018,2018,2016,Y,107
3,63669288,500783193,US,A,2018-05-07,2018,2018,2016,Y,107
4,63669288,501667906,US,A,2018-05-07,2018,2018,2016,Y,107
5,63669288,500783039,US,A,2018-05-07,2018,2018,2016,Y,107
6,63669288,500783854,US,A,2018-05-07,2018,2018,2016,N,107
7,63669288,500788490,US,A,2018-05-07,2018,2018,2016,N,107
8,63669288,500778685,US,A,2018-05-07,2018,2018,2016,Y,107
9,63669288,500783701,US,A,2018-05-07,2018,2018,2016,Y,107


## Year

In [13]:
years = example_family[["appln_id", "appln_auth", "appln_filing_year", "earliest_publn_year", "earliest_filing_year"]]
family_year = example_family["earliest_filing_year"].min()
print(f"Family {example_family_id}: {len(example_family)} applications filed between "
      f"{example_family['appln_filing_year'].min()} and {example_family['appln_filing_year'].max()}; "
      f"family year (earliest priority) = {family_year}")
years

Family 63669288: 59 applications filed between 2018 and 2019; family year (earliest priority) = 2016


,appln_id,appln_auth,appln_filing_year,earliest_publn_year,earliest_filing_year
0,500778272,US,2018,2018,2016
1,500783187,US,2018,2018,2016
2,500782752,US,2018,2018,2016
3,500783193,US,2018,2018,2016
4,501667906,US,2018,2018,2016
5,500783039,US,2018,2018,2016
6,500783854,US,2018,2018,2016
7,500788490,US,2018,2018,2016
8,500778685,US,2018,2018,2016
9,500783701,US,2018,2018,2016


In [15]:
appln_ids = df_ranked["appln_id"].dropna().astype(int).tolist()
CHUNK_SIZE = 25000
auth_dfs = []

for i in range(0, len(appln_ids), CHUNK_SIZE):
    chunk_ids = appln_ids[i : i + CHUNK_SIZE]
    q_auth = db.query(
        TLS201_APPLN.appln_id,
        TLS201_APPLN.appln_auth
    ).filter(TLS201_APPLN.appln_id.in_(chunk_ids))
    
    auth_dfs.append(patstat.df(q_auth))

auth_mapping = pd.concat(auth_dfs, ignore_index=True)

# Merge into df_ranked and update target subset
df_ranked = df_ranked.merge(auth_mapping, on="appln_id", how="left")
final_enel_dataset = df_ranked[df_ranked["sbert_similarity"] >= SIMILARITY_THRESHOLD].copy()

# 2. Flag PCT Applications and Families
final_enel_dataset["is_pct"] = final_enel_dataset["appln_auth"] == "WO"

pct_families = set(final_enel_dataset[final_enel_dataset["is_pct"]]["docdb_family_id"].dropna())
final_enel_dataset["family_is_pct"] = final_enel_dataset["docdb_family_id"].isin(pct_families)

# 3. Print PCT shares for technical appendix
pct_app_share = (final_enel_dataset["is_pct"].sum() / len(final_enel_dataset)) * 100
pct_fam_share = (len(pct_families) / final_enel_dataset["docdb_family_id"].nunique()) * 100

print(f"\n PCT INTERNATIONAL FILINGS:")
print(f"  • PCT Applications (WO authority): {final_enel_dataset['is_pct'].sum():,} ({pct_app_share:.1f}%)")
print(f"  • PCT Families (at least 1 WO filing): {len(pct_families):,} ({pct_fam_share:.1f}%)")


 PCT INTERNATIONAL FILINGS:
  • PCT Applications (WO authority): 3,252 (6.6%)
  • PCT Families (at least 1 WO filing): 3,230 (7.9%)
